In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from scipy.stats import norm
from copy import deepcopy
from scipy.stats import t as t_dist
import numpy as np
from scipy.stats import ttest_ind

### Compare with Benjamini–Hochberg procedure

In [ ]:
K = 100
N = 10
Z = norm.ppf(0.975)
std = 3

cost = np.zeros((4,5,1000))
p_value_matrix = np.zeros((K,1000))
p_value_matrix_shrunken = np.zeros((K,1000))


for index1,anchor_mean in enumerate([1,2,3,4,5]):
    for time_seq in tqdm(range(1000)):
        true_tao = np.random.normal(anchor_mean,3, K)
        optimal_cost = np.sum(true_tao[np.argwhere(true_tao>0)])    
        X = np.zeros((K,N))
        for i in range(K):
            random_set = np.random.choice(range(N), 5, replace=False)
            for j in random_set:
                X[i,j] = 1
        Y = np.zeros((K,N))
        for i in range(K):
            for j in range(N):
                Y[i,j] = true_tao[i] * X[i,j] + np.random.normal(0,std)

        #DM
        tao_hat = np.zeros(K)
        variance = np.zeros(K)
        #variance1 = np.zeros(K)
        p_value_list = np.zeros(K)
        for k in range(K):
            group_1 = Y[k,X[k,:]==1]
            group_0 = Y[k,X[k,:]==0]
            t_stat, p_value = ttest_ind(group_1, group_0, equal_var = False)  
            diff_mean = group_1.mean() - group_0.mean()
            tao_hat[k] = diff_mean
            p_value_list[k] = p_value
            variance[k] = N*(group_1.var(ddof=1) / len(group_1) + group_0.var(ddof=1) / len(group_0))
        p_value_matrix[:,time_seq] = p_value_list
            #tao_hat[i] = np.sum(Y[i,:]*X[i,:])/np.sum(X[i,:]) - np.sum(Y[i,:]*(1-X[i,:]))/np.sum(1-X[i,:])
            #variance[i] = np.sum((Y[i,:]-tao_hat[i]*X[i,:])**2)/(N)
        
        #Benjamini–Hochberg procedure
        decision4 = []
        sorted_indices = np.argsort(p_value_list)
        max_index = -1
        for i,index in enumerate(sorted_indices):
            p_value_k = p_value_list[index]
            if p_value_k <= (i+1)*0.05/K:
                max_index = i
        for i,index in enumerate(sorted_indices):
            if i <= max_index and tao_hat[index] > 0:
                decision4.append(index)  


        tao_0 = np.mean(tao_hat)
        numerator = np.mean(variance)
        denumerator = np.mean((tao_hat - tao_0)**2) - numerator/N

        bayesian_tao = np.zeros(K)
        bayesian_beta = np.zeros(K)
        #p_value_list_bayesian = np.zeros(K)
        decision3 = []
        # #bayesian_variance = np.zeros(K)
        for k in range(K):
            if denumerator <= 0:
                theta = 1
                posteri_mean = tao_hat[k]*theta + (1 - theta)*tao_0
                posteri_var = 1/(N/variance[k])
                dist = norm(loc=posteri_mean, scale=np.sqrt(posteri_var))
                prob = dist.sf(0)  # survival function: P(X > x)
                if prob > 1 - 0.025:
                    decision3.append(k)
            else:
                bayesian_beta[k] = max(variance[k]/denumerator,0)
                theta = N/(N+bayesian_beta[k])
                posteri_mean = tao_hat[k]*theta + (1 - theta)*tao_0
                posteri_var = 1/(1/denumerator+ N/variance[k])
                dist = norm(loc=posteri_mean, scale=np.sqrt(posteri_var))

                prob = dist.sf(0)  # survival function: P(X > x)
                if prob > 1 - 0.025:
                    decision3.append(k)

        
        
        if anchor_mean != 0:
            beta = numerator/denumerator + Z*np.sqrt(N*numerator)/tao_0
            beta = max(0,beta)
            theta = N/(N+beta)
            #shunken_saa
            tao_shunken_hat = np.zeros(K)
            p_value_list_shrunken = np.zeros(K)
            #anchor_mean = np.mean(tao_hat)
            #theta = np.sum((true_tao - anchor_mean)**2)/(np.sum((true_tao - anchor_mean)**2)+K*50**2*2/N)
            #theta = (N/2)/((N/2) + 50**2/9)
            Y_shunken = deepcopy(Y)
            Y_shunken = theta*Y_shunken 
            for k in range(K):
                group_1 = Y_shunken[k,X[k,:]==1] + (1-theta)*tao_0
                group_0 = Y_shunken[k,X[k,:]==0]
                t_stat, p_value = ttest_ind(group_1, group_0, equal_var = False)  
                diff_mean1 = group_1.mean() - group_0.mean()
                tao_shunken_hat[k] = diff_mean1
                p_value_list_shrunken[k] = p_value
            
            p_value_matrix_shrunken[:,time_seq] = p_value_list_shrunken
            # cost[0,index1,time_seq] = np.sum(true_tao[np.argwhere(tao_hat>(Z*np.sqrt(variance))/np.sqrt(N))])/optimal_cost
            # cost[1,index1,time_seq] = np.sum(true_tao[np.argwhere(tao_shunken_hat>(Z*np.sqrt(variance)*theta)/np.sqrt(N))])/optimal_cost
            decision1 = np.intersect1d(np.argwhere(p_value_list<0.05), np.argwhere(tao_hat>0))
            decision2 = np.intersect1d(np.argwhere(p_value_list_shrunken<0.05), np.argwhere(tao_shunken_hat>0))
            #decision3 = np.intersect1d(np.argwhere(p_value_list_bayesian<0.05), np.argwhere(bayesian_tao>0))
            cost[0,index1,time_seq] = np.sum(true_tao[decision1])/optimal_cost
            cost[1,index1,time_seq] = np.sum(true_tao[decision2])/optimal_cost
            cost[2,index1,time_seq] = np.sum(true_tao[decision3])/optimal_cost
            cost[3,index1,time_seq] = np.sum(true_tao[decision4])/optimal_cost
            #cost[3,index1,time_seq] = np.sum(true_tao[decision4])/optimal_cost
            #cost[3,index1,time_seq] = np.sum(true_tao[decision4])/optimal_cost
            
            
            

In [ ]:
plt.figure(figsize=(8,6))
plt.xlabel(r'The value of $\tau_0$',fontsize=15)
plt.ylabel('Optimality Ratio (OR)',fontsize=15)
x = [1,2,3,4,5]
y3 = np.mean(cost[0],axis=1)
plt.plot(x, y3, color = '#8CA5EA',linestyle = '--', marker='o',label = "ITR")

y3 = np.mean(cost[1],axis=1)
plt.plot(x, y3, color = '#495373',marker = "s",label = "DPTR")

y3 = np.mean(cost[2],axis=1)
plt.plot(x, y3, color = '#E3738B',marker = "s",label = "Bayesian",linestyle = '--')

y4 = np.mean(cost[3],axis=1)
plt.plot(x, y4, color = '#A569BD',marker = "^",label = "BH procedure")

plt.xticks([1,2,3,4,5],fontsize=15)
plt.yticks(fontsize=15)
plt.legend(fontsize=15)
plt.grid()
plt.savefig('scenario1_figures/performance_compare_with_BH.png',dpi=300,bbox_inches = 'tight')
plt.show()